# Patent Applicant Name Harmonization — Stages 2 & 3: Domain Rectification & Anchor Consolidation

This notebook consumes the domain-agnostic Stage 1 lookup (`outputs/harmonization_lookup_stage1.csv`) alongside the raw PATSTAT table (`../qc_applicants_PROD.csv`) to refine applicant clusters for single-sector patent datasets (e.g., Quantum Computing).

### Refinement Architecture
1. **Stage 2 — Sector-Stratified Hairball Rectification:** Audits every multi-member Stage 1 cluster (`MIN_CLUSTER_SIZE_TO_SPLIT = 2`), enforces consistent `entity_type` boundaries (`company`, `university`, `gov / non-profit`, `uncertain`), strips domain/institutional/geographic boilerplate (`BOILERPLATE_CORE`), weights distinctive tokens by $\text{idf}(t)^2$, and re-partitions mixed or chained clusters via high-resolution Louvain (`resolution = 1.35`).
2. **Stage 3 — Core-Anchor & Subsidiary Consolidation:** Recovers recall across regional subsidiaries, word-order swaps, and exact acronym expansions (`IBM` $\leftrightarrow$ `IBM DEUTSCHLAND` / `INTERNATIONAL BUSINESS MACHINES`, `STANFORD UNIV` $\leftrightarrow$ `LELAND STANFORD JUNIOR UNIV`) within compatible entity categories.
3. **Final Production Export:** Outputs `outputs/harmonization_lookup_rectified.csv` and left-joins it onto the complete PATSTAT extract (`outputs/qc_applicants_harmonized_PROD.csv`).

## 1. Imports & Rectification Configuration

Defines input/output paths in `outputs/` and `..`, Stage 2 hairball-splitting cutoffs, and Stage 3 rare-anchor consolidation thresholds.

In [ ]:
import os
from collections import Counter
import numpy as np
import pandas as pd
import networkx as nx
import networkx.algorithms.community as nx_comm
from rapidfuzz.fuzz import token_sort_ratio as rf_token_sort
from sentence_transformers import SentenceTransformer

# --- Paths & Folder Setup ---
INPUT_FOLDER = ".."  # parent directory containing qc_applicants_PROD.csv
OUTPUT_FOLDER = "outputs"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

INPUT_RAW_CSV = os.path.join(INPUT_FOLDER, "qc_applicants_PROD.csv")
INPUT_STAGE1_LOOKUP = os.path.join(OUTPUT_FOLDER, "harmonization_lookup_stage1.csv")
OUTPUT_FINAL_LOOKUP = os.path.join(OUTPUT_FOLDER, "harmonization_lookup_rectified.csv")
OUTPUT_FINAL_DATA = os.path.join(OUTPUT_FOLDER, "qc_applicants_harmonized_PROD.csv")

NAME_COL = "psn_name"
WEIGHT_COL = None  # optional: e.g. "n_families" if available

# --- Stage 2: Hairball Rectification Parameters ---
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
MIN_CLUSTER_SIZE_TO_SPLIT = 2
CORE_JAC_MIN = 0.35
CORE_EDIT_MIN = 0.80
SIM_FLOOR_S2 = 0.56
LOUVAIN_RESOLUTION_S2 = 1.35
LOUVAIN_SEED = 42

# --- Stage 3: Core-Anchor Consolidation Parameters ---
RARE_ANCHOR_MAX_DF = 12.0
CORE_FUZZY_MIN = 90.0

## 2. Load Stage 1 Lookup, Map Entity Types & Compute Corpus Statistics

Loads `outputs/harmonization_lookup_stage1.csv` and `../qc_applicants_PROD.csv`, maps `psn_sector_imputed` into canonical `entity_type` categories (`company`, `university`, `gov / non-profit`, `uncertain`), enforces a single majority `entity_type` per unique `clean` string, and computes PATSTAT filing weights and corpus-wide token IDF statistics.

In [ ]:
SECTOR_TO_ENTITY = {
    "COMPANY": "company",
    "UNIVERSITY": "university",
    "GOV NON-PROFIT": "gov / non-profit",
    "UNCERTAIN": "uncertain",
    "INDIVIDUAL": "individual",
}

raw_df = pd.read_csv(INPUT_RAW_CSV)
df_work = pd.read_csv(INPUT_STAGE1_LOOKUP)

df_work["entity_type"] = (
    df_work["psn_sector_imputed"].str.strip().str.upper().map(SECTOR_TO_ENTITY).fillna("uncertain")
)

df_work = df_work[
    (df_work["entity_type"] != "individual") &
    (df_work["clean"].notna()) &
    (df_work["clean"].str.strip() != "")
].copy()

# Enforce consistent entity_type per unique cleaned string (prevents RIKEN / OXFORD UNIV INNOVATION splits)
clean_maj_type = (
    df_work.groupby(["clean", "entity_type"]).size()
    .reset_index(name="cnt")
    .sort_values(["clean", "cnt"], ascending=[True, False])
    .drop_duplicates("clean")
    .set_index("clean")["entity_type"]
)
df_work["entity_type"] = df_work["clean"].map(clean_maj_type)

# Compute PATSTAT filing weights per cleaned string
raw_clean_map = df_work.drop_duplicates(NAME_COL).set_index(NAME_COL)["clean"]
if WEIGHT_COL and WEIGHT_COL in raw_df.columns:
    tmp_w = raw_df.assign(clean=raw_df[NAME_COL].map(raw_clean_map)).dropna(subset=["clean"])
    name_weights = tmp_w.groupby("clean")[WEIGHT_COL].sum().to_dict()
else:
    name_weights = raw_df[NAME_COL].map(raw_clean_map).dropna().value_counts().to_dict()

unique_clean = sorted(df_work["clean"].unique())
N_CORPUS = len(unique_clean)
df_counts = Counter(t for s in unique_clean for t in set(s.split()))
idf_map = {t: np.log(N_CORPUS / c) for t, c in df_counts.items()}

# Encode unique cleaned names for Stage 2 intra-cluster cosine verification
model = SentenceTransformer(MODEL_NAME)
emb = model.encode(unique_clean, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
emb_map = dict(zip(unique_clean, emb))

print(f"Loaded {len(df_work):,} organizational name variants ({N_CORPUS:,} unique cleaned names) across {df_work['harm_id'].nunique():,} Stage 1 clusters.")
print(df_work.groupby("entity_type").agg(n_raw=(NAME_COL, "nunique"), n_clean=("clean", "nunique")))

## 3. Stage 2: Sector-Stratified Hairball Rectification

Audits every Stage 1 cluster with `size >= 2` (`MIN_CLUSTER_SIZE_TO_SPLIT = 2`) and breaks apart transitive chain-merges caused by shared sector/geographic templates (`HONDA RESEARCH INST EUROPE` + `TOSHIBA RESEARCH EUROPE`, `WILLIS GROUP` + `INSPUR GROUP`, `SIEMENS MOBILITY` + `TENCENT MOBILITY`, `UNIV DEGLI STUDI MILANO` + `UNIV DEGLI STUDI BOLOGNA`) using:
1. **Strict `entity_type` isolation** inside every cluster.
2. **Squared-IDF core token Jaccard (`CORE_JAC_MIN = 0.35`)** after stripping institutional, sector, and geographic boilerplate (`BOILERPLATE_CORE`).
3. **Token-sorted edit guard (`CORE_EDIT_MIN = 0.80`)** and high-resolution Louvain partitioning (`resolution = 1.35`).

In [ ]:
BOILERPLATE_CORE = {
    # Institutional & Legal
    "UNIV", "INST", "INSTITUTE", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE",
    "POLITECNICA", "POLYTECHNICAL", "POLYTECHNIC", "UNIVERSITETT", "UNIVERSITAET", "UNIVERSITAT",
    "UNIVERSITE", "UNIVERSIDAD", "UNIVERSITA", "UNIVERSITET", "TECHNISCHE", "VOCATIONAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "COLTD",
    "HOLDINGS", "INTERNATIONAL", "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL",
    "CENTRAL", "FEDERAL", "LIMITED", "RESPONSIBILITY", "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS",
    "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION", "PRESIDENT", "FELLOWS",
    "HOSPITAL", "DEPARTMENT", "MINISTRY", "GOVERNMENT", "AGENCY", "SECRETARY", "DIRECTOR",
    "COMMITTEE", "COUNCIL", "INSTITUTION", "ADMINISTRATIVE", "INDEPENDENT",
    "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS", "COOPERATION", "ACADEMIC", "ACADEMIA",
    "FEDERALNOE", "GOSUDARSTVENNOE", "BYUDZHETNOE", "OBRAZOVATELNOE", "UCHREZHDENIE", "VYSSHEGO", "OBRAZOVANIYA",
    "GESELLSCHAFT", "FORDERUNG", "ANGEWANDTEN", "FORSCHUNG", "WISSENSCHAFTEN", "EV",
    # Sector & Business Descriptors
    "QUANTUM", "QUANTIQUE", "COMPUTING", "COMPUTER", "CALCULATING", "INFORMATION", "SYSTEMS", "SYSTEM",
    "NETWORK", "NETWORKS", "ELECTRONIC", "ELECTRONICS", "ELECTRIC", "POWER", "GRID", "ENERGY",
    "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE", "SOLUTIONS", "SOLUTION",
    "INFRASTRUCTURE", "INSTRUMENT", "INSTRUMENTS", "DEVICES", "LICENSING", "PERMISSION",
    "INNOVATION", "INNOVATIONS", "VENTURES", "CAPITAL", "INVESTMENT", "PORTFOLIO",
    "INTELLECTUAL", "PROPERTY", "FINANCIAL", "INDUSTRY", "INDUSTRIAL", "DEVELOPMENT",
    "TECHNOLOGY", "MANUFACTURING", "APPLICATION", "APPLIED", "SERVICES", "SERVICE",
    "MACHINES", "EQUIPMENT", "AEROSPACE", "SPACE", "VISION", "BIOLOGICAL", "HEALTH",
    "MEDICINE", "MEDICAL", "ATOMIC", "ALTERNATIVE", "MOBILE", "MOBILITY", "TELECOM",
    "TELECOMMUNICATIONS", "COMMUNICATION", "COMMUNICATIONS", "ENGINEERING", "AGRICULTURAL",
    "MARITIME", "PETROLEUM", "MINERALS", "JIAOTONG", "EDUCATION", "HIGHER", "OPTICS",
    "OPTO", "PHOTONICS", "NANO", "NANOSCIENCE", "NANOTECHNOLOGY", "TOOLS", "CLOUD",
    "STORAGE", "SECURITY", "DEFENCE", "DEFENSE", "MILITARY", "PLA", "ARMY", "NAVY", "FORCE",
    "CIRCUITS", "INTEGRATED", "TIME", "PLATFORMS", "PLATFORM", "MOTOR", "INTELLIGENCE", "ARTIFICIAL",
    # Countries, Regions & Major Cities
    "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "AMERICA", "AMERICAN",
    "UNITED", "STATES", "KINGDOM", "UK", "US", "DEUTSCHLAND", "GERMANY", "FRANCE", "AUSTRALIA",
    "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG",
    "SICHUAN", "HUBEI", "HUNAN", "HENAN", "HEBEI", "FUJIAN", "SHANDONG", "SHANXI",
    "SHAANXI", "LIAONING", "JILIN", "HEILONGJIANG", "YUNNAN", "GUIZHOU", "GANSU",
    "QINGHAI", "HAINAN", "NINGXIA", "XINJIANG", "XIZANG", "GUANGXI", "INNER", "MONGOLIA",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN",
    "HARBIN", "TIANJIN", "XIAN", "CHANGCHUN", "SHENYANG", "ZHENGZHOU", "CHANGSHA",
    "FUZHOU", "XIAMEN", "JINAN", "KUNMING", "TAIYUAN", "DONGGUAN", "QINGDAO", "WUXI",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST",
    "NEW", "COLORADO", "FLORIDA", "ARIZONA", "MICHIGAN", "OHIO", "PENN", "TEXAS", "CALIFORNIA",
    "YORK", "WASHINGTON", "VIRGINIA"
}


def _core_tokens(s: str) -> set:
    toks = str(s).split()
    core = {t for t in toks if t not in BOILERPLATE_CORE and len(t) >= 2}
    return core if core else set(toks)


def _idf_jaccard_core(a: str, b: str) -> float:
    sa, sb = _core_tokens(a), _core_tokens(b)
    if not sa or not sb:
        return 0.0
    inter = sa & sb
    union = sa | sb
    w_inter = sum((idf_map.get(t, 1.0)) ** 2 for t in inter)
    w_union = sum((idf_map.get(t, 1.0)) ** 2 for t in union)
    return float(w_inter / w_union) if w_union > 0 else 0.0


def rectify_stage1_cluster(members_df: pd.DataFrame) -> list:
    '''Split a Stage 1 cluster by entity_type and tightened core-lexical + semantic graph.'''
    sub_clusters = []
    for etype, grp in members_df.groupby("entity_type"):
        u_names = sorted(grp["clean"].unique())
        if len(u_names) < MIN_CLUSTER_SIZE_TO_SPLIT or etype == "uncertain":
            for u in u_names:
                sub_clusters.append((etype, [u]))
            continue

        G_sub = nx.Graph()
        G_sub.add_nodes_from(u_names)
        for i in range(len(u_names)):
            for j in range(i + 1, len(u_names)):
                a, b = u_names[i], u_names[j]
                cos = float(np.dot(emb_map[a], emb_map[b]))
                jac_core = _idf_jaccard_core(a, b)
                ed_sort = rf_token_sort(a, b) / 100.0
                if jac_core < CORE_JAC_MIN and ed_sort < CORE_EDIT_MIN:
                    continue
                score = 0.34 * cos + 0.33 * jac_core + 0.33 * ed_sort
                if score >= SIM_FLOOR_S2:
                    G_sub.add_edge(a, b, weight=score)

        comms = nx_comm.louvain_communities(G_sub, weight="weight", resolution=LOUVAIN_RESOLUTION_S2, seed=LOUVAIN_SEED)
        for c in comms:
            sub_clusters.append((etype, sorted(c)))
    return sub_clusters


stage2_records = []
for hid, grp in df_work.groupby("harm_id"):
    for etype, members in rectify_stage1_cluster(grp):
        stage2_records.append((etype, members))

stage2_records.sort(key=lambda item: (-sum(name_weights.get(m, 1) for m in item[1]), min(item[1])))

clean_to_id_s2 = {}
clean_to_canon_s2 = {}
for idx, (etype, members) in enumerate(stage2_records, start=1):
    new_id = f"HARM_{idx:05d}"
    best_canon = min(members, key=lambda s: (-name_weights.get(s, 1), len(s), s))
    for m in members:
        clean_to_id_s2[m] = new_id
        clean_to_canon_s2[m] = best_canon

df_work["harm_id_final"] = df_work["clean"].map(clean_to_id_s2)
df_work["harmonized_name_final"] = df_work["clean"].map(clean_to_canon_s2)

print(f"Stage 2 Rectification complete: {df_work['harm_id'].nunique():,} -> {df_work['harm_id_final'].nunique():,} clusters.")

## 4. Stage 3: Core-Anchor & Subsidiary Consolidation

Re-merges split regional subsidiaries, word-order permutations, and exact acronym expansions **within compatible entity categories** when canonical names satisfy any of three symbolic rules:
1. **Rare Brand-Anchor Subset:** One cluster's distinguishing tokens are a strict subset of another's and share at least one rare proper identifier ($\text{df} \le 12$), preserving US state names (`CALIFORNIA`, `TEXAS`, `FLORIDA`, etc.) and institutional types (`UNIV` vs. `POLYTECHNIC`) as distinct academic identifiers.
2. **Core Token-Sorted Fuzzy Match:** Distinguishing cores ($\ge 8$ characters) achieve $\ge 90\%$ `token_sort_ratio`.
3. **Exact Acronym Expansion:** A 3–6 letter uppercase acronym matches the exact word initials of another entity in the same sector (`IBM` $\leftrightarrow$ `INTERNATIONAL BUSINESS MACHINES`).

In [ ]:
US_STATES = {"COLORADO", "FLORIDA", "ARIZONA", "MICHIGAN", "OHIO", "PENN", "TEXAS", "CALIFORNIA", "WASHINGTON", "VIRGINIA"}

ANCHOR_STOPWORDS = {
    # Institutional & Legal
    "UNIV", "INST", "INSTITUTE", "COLLEGE", "ACADEMY", "SCHOOL", "HOCHSCHULE", "ECOLE",
    "POLITECNICA", "POLYTECHNICAL", "POLYTECHNIC", "UNIVERSITETT", "UNIVERSITAET", "UNIVERSITAT",
    "UNIVERSITE", "UNIVERSIDAD", "UNIVERSITA", "UNIVERSITET", "TECHNISCHE", "VOCATIONAL",
    "TECH", "SCIENCE", "RESEARCH", "LAB", "CENTER", "GROUP", "CORP", "COMPANY", "COLTD",
    "HOLDINGS", "INTERNATIONAL", "NATIONAL", "STATE", "PUBLIC", "ROYAL", "NORMAL", "GENERAL",
    "CENTRAL", "FEDERAL", "LIMITED", "RESPONSIBILITY", "KORPERSCHAFT", "OFFENTLICHEN", "RECHTS",
    "BOARD", "REGENTS", "TRUSTEES", "FOUNDATION", "ASSOCIATION", "PRESIDENT", "FELLOWS",
    "HOSPITAL", "DEPARTMENT", "MINISTRY", "GOVERNMENT", "AGENCY", "SECRETARY", "DIRECTOR",
    "COMMITTEE", "COUNCIL", "INSTITUTION", "ADMINISTRATIVE", "INDEPENDENT",
    "FEDERALNOE", "GOSUDARSTVENNOE", "BYUDZHETNOE", "OBRAZOVATELNOE", "UCHREZHDENIE", "VYSSHEGO", "OBRAZOVANIYA",
    "GESELLSCHAFT", "FORDERUNG", "ANGEWANDTEN", "FORSCHUNG", "WISSENSCHAFTEN", "EV",
    "DEGLI", "STUDI", "NORMALE", "SUPERIEURE", "POSTS", "COOPERATION", "ACADEMIC", "ACADEMIA",
    # Sector & Business Descriptors
    "QUANTUM", "QUANTIQUE", "COMPUTING", "COMPUTER", "CALCULATING", "INFORMATION", "SYSTEMS", "SYSTEM",
    "NETWORK", "NETWORKS", "ELECTRONIC", "ELECTRONICS", "ELECTRIC", "POWER", "GRID", "ENERGY",
    "DIGITAL", "INTELLIGENT", "SMART", "DATA", "SOFTWARE", "SOLUTIONS", "SOLUTION",
    "INFRASTRUCTURE", "INSTRUMENT", "INSTRUMENTS", "DEVICES", "LICENSING", "PERMISSION",
    "INNOVATION", "INNOVATIONS", "VENTURES", "CAPITAL", "INVESTMENT", "PORTFOLIO",
    "INTELLECTUAL", "PROPERTY", "FINANCIAL", "INDUSTRY", "INDUSTRIAL", "DEVELOPMENT",
    "TECHNOLOGY", "MANUFACTURING", "APPLICATION", "APPLIED", "SERVICES", "SERVICE",
    "MACHINES", "EQUIPMENT", "AEROSPACE", "SPACE", "VISION", "BIOLOGICAL", "HEALTH",
    "MEDICINE", "MEDICAL", "ATOMIC", "ALTERNATIVE", "MOBILE", "MOBILITY", "TELECOM",
    "TELECOMMUNICATIONS", "COMMUNICATION", "COMMUNICATIONS", "ENGINEERING", "AGRICULTURAL",
    "MARITIME", "PETROLEUM", "MINERALS", "JIAOTONG", "EDUCATION", "HIGHER", "OPTICS",
    "OPTO", "PHOTONICS", "NANO", "NANOSCIENCE", "NANOTECHNOLOGY", "TOOLS", "CLOUD",
    "STORAGE", "SECURITY", "DEFENCE", "DEFENSE", "MILITARY", "PLA", "ARMY", "NAVY", "FORCE",
    "MOTOR", "PLATFORMS", "PLATFORM", "INTELLIGENCE", "ARTIFICIAL", "CIRCUITS", "INTEGRATED", "TIME",
    "SILICON", "WAVE", "GLOBAL", "STAR", "GREAT", "SUPER", "ADVANCED", "FUTURE", "OPEN", "SECOND", "MATRIX",
    # Countries, Regions & Major Cities
    "CHINA", "CHINESE", "JAPAN", "KOREA", "EUROPE", "EUROPEAN", "AMERICA", "AMERICAN",
    "UNITED", "STATES", "KINGDOM", "UK", "US", "DEUTSCHLAND", "GERMANY", "FRANCE", "AUSTRALIA",
    "CANADA", "ISRAEL", "SWITZERLAND", "NETHERLANDS",
    "BEIJING", "SHANGHAI", "SHENZHEN", "GUANGDONG", "ANHUI", "JIANGSU", "ZHEJIANG",
    "SICHUAN", "HUBEI", "HUNAN", "HENAN", "HEBEI", "FUJIAN", "SHANDONG", "SHANXI",
    "SHAANXI", "LIAONING", "JILIN", "HEILONGJIANG", "YUNNAN", "GUIZHOU", "GANSU",
    "QINGHAI", "HAINAN", "NINGXIA", "XINJIANG", "XIZANG", "GUANGXI", "INNER", "MONGOLIA",
    "HEFEI", "NANJING", "SUZHOU", "HANGZHOU", "WUHAN", "CHENGDU", "CHONGQING", "DALIAN",
    "HARBIN", "TIANJIN", "XIAN", "CHANGCHUN", "SHENYANG", "ZHENGZHOU", "CHANGSHA",
    "FUZHOU", "XIAMEN", "JINAN", "KUNMING", "TAIYUAN", "DONGGUAN", "QINGDAO", "WUXI",
    "NORTH", "SOUTH", "EAST", "WEST", "NORTHEAST", "NORTHWEST", "SOUTHEAST", "SOUTHWEST",
    "NEW", "YORK", "OXFORD", "CAMBRIDGE", "TOKYO", "SEOUL", "PARIS", "BERLIN", "MUNICH",
    "MUNCHEN", "ZURICH", "VIENNA", "WIEN", "DELFT"
} | US_STATES

INST_TYPE_TOKENS = {"UNIV", "POLYTECHNIC", "COLLEGE", "SCHOOL", "MINES", "INST", "ACADEMY", "HOSPITAL"}


def _extract_anchor(name: str, etype: str) -> set:
    stops = (ANCHOR_STOPWORDS - US_STATES) if etype == "university" else ANCHOR_STOPWORDS
    return {t for t in str(name).split() if t not in stops and len(t) >= 3}


def _is_exact_acronym(acro: str, full_str: str) -> bool:
    if not (3 <= len(acro) <= 6 and acro.isalpha() and len(acro.split()) == 1):
        return False
    words = [w for w in full_str.split() if w != acro]
    return len(words) == len(acro) and acro == "".join(w[0] for w in words)


stage2_clusters = (
    df_work.groupby(["harm_id_final", "entity_type", "harmonized_name_final"])
    .agg(weight=("clean", lambda s: sum(name_weights.get(x, 1) for x in s)))
    .reset_index()
)

RARE_ANCHOR_IDF = np.log(N_CORPUS / RARE_ANCHOR_MAX_DF)
G_anchor = nx.Graph()
G_anchor.add_nodes_from(stage2_clusters["harm_id_final"])

for etype, grp in stage2_clusters.groupby("entity_type"):
    if etype == "uncertain":
        continue
    rows = list(grp.itertuples())
    for i in range(len(rows)):
        a = rows[i].harmonized_name_final
        ca = _extract_anchor(a, etype)
        ca_str = " ".join(sorted(ca))
        ia = set(a.split()) & INST_TYPE_TOKENS
        for j in range(i + 1, len(rows)):
            b = rows[j].harmonized_name_final
            cb = _extract_anchor(b, etype)
            cb_str = " ".join(sorted(cb))
            ib = set(b.split()) & INST_TYPE_TOKENS

            if etype == "university" and ia and ib and ia != ib:
                continue

            do_merge = False
            if ca and cb and (ca <= cb or cb <= ca):
                shared = ca & cb
                if any(idf_map.get(t, 0.0) >= RARE_ANCHOR_IDF for t in shared):
                    do_merge = True

            if not do_merge and len(ca_str) >= 8 and len(cb_str) >= 8:
                if rf_token_sort(ca_str, cb_str) >= CORE_FUZZY_MIN:
                    do_merge = True

            if not do_merge and (_is_exact_acronym(a, b) or _is_exact_acronym(b, a)):
                do_merge = True

            if do_merge:
                G_anchor.add_edge(rows[i].harm_id_final, rows[j].harm_id_final)

merged_components = list(nx.connected_components(G_anchor))
old_id_to_weight = stage2_clusters.set_index("harm_id_final")["weight"].to_dict()
old_id_to_canon = stage2_clusters.set_index("harm_id_final")["harmonized_name_final"].to_dict()

merged_components.sort(
    key=lambda comp: (-sum(old_id_to_weight[hid] for hid in comp), min(old_id_to_canon[hid] for hid in comp))
)

old_to_new_id = {}
for idx, comp in enumerate(merged_components, start=1):
    new_hid = f"HARM_{idx:05d}"
    for hid in comp:
        old_to_new_id[hid] = new_hid

df_work["harm_id_final"] = df_work["harm_id_final"].map(old_to_new_id)


def _pick_canonical(sub_clean: pd.Series) -> str:
    unique_members = sub_clean.unique()
    return min(unique_members, key=lambda s: (-name_weights.get(s, 1), len(s), s))


canon_map = df_work.groupby("harm_id_final")["clean"].apply(_pick_canonical)
df_work["harmonized_name_final"] = df_work["harm_id_final"].map(canon_map)

print(f"Stage 3 Anchor Consolidation complete: {len(stage2_clusters):,} -> {df_work['harm_id_final'].nunique():,} final entities.")

## 5. Final Production Export & Quality Diagnostics

Exports:
* **`outputs/harmonization_lookup_rectified.csv`:** Final deduplicated organizational mapping (`psn_name` $\rightarrow$ `clean`, `entity_type`, `harm_id_final`, `harmonized_name_final`).
* **`outputs/qc_applicants_harmonized_PROD.csv`:** The full raw PATSTAT table (`../qc_applicants_PROD.csv`) left-joined with the rectified entity mapping so all organizational rows (including imputed `UNKNOWN` rows) are harmonized while individual inventor rows remain intact with `NaN` entity IDs.

In [ ]:
lookup_out = (
    df_work[[NAME_COL, "clean", "entity_type", "harm_id_final", "harmonized_name_final"]]
    .drop_duplicates(subset=[NAME_COL])
    .sort_values(["harm_id_final", NAME_COL])
)
lookup_out.to_csv(OUTPUT_FINAL_LOOKUP, index=False)

full_harmonized = raw_df.merge(lookup_out, on=NAME_COL, how="left")
full_harmonized.to_csv(OUTPUT_FINAL_DATA, index=False)

n_org_rows = full_harmonized["harm_id_final"].notna().sum()
n_ind_rows = full_harmonized["harm_id_final"].isna().sum()

print(f"Saved lookup table -> {OUTPUT_FINAL_LOOKUP} ({len(lookup_out):,} unique org names -> {lookup_out['harm_id_final'].nunique():,} entities)")
print(f"Saved full dataset -> {OUTPUT_FINAL_DATA} ({len(full_harmonized):,} total rows: {n_org_rows:,} org rows harmonized, {n_ind_rows:,} individual/empty rows)")

print("\n--- Consolidation Summary by Entity Type ---")
summary = lookup_out.groupby("entity_type").agg(
    n_raw=(NAME_COL, "nunique"),
    n_clean=("clean", "nunique"),
    n_entities=("harm_id_final", "nunique"),
)
summary["consolidation_%"] = (100 * (1 - summary["n_entities"] / summary["n_clean"])).round(1)
print(summary.to_string())

print("\n--- Top 15 Largest Harmonized Entities (by patent rows) ---")
print(full_harmonized["harmonized_name_final"].value_counts().head(15).to_string())